# Hands-On 6: CrewAI Memory and Knowledge

### Definition
**Memory** lets agents retain useful context during and across workflow executions.

**Knowledge** gives agents access to information stored in documents or other sources.

### Goal
- Configure crew memory
- Create a knowledge source
- Add a document
- Ask contextual questions
- Reuse previous information
- Compare memory-enabled and memory-disabled workflows

In [1]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"))
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

OpenRouter key loaded: True
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


In [2]:
llm = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")
print("LLM:", llm)

LLM: openai/gpt-4o-mini


## 1. Create Context-Aware Agents

The agents are designed to use information from earlier tasks instead of asking for the same information again.

In [3]:
researcher = Agent(
    role="Context-Aware Researcher",
    goal="Research topics while maintaining context from previous tasks",
    backstory="A research specialist who carefully tracks names, facts, and decisions from earlier work.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role="Contextual Writer",
    goal="Build new content using information already established",
    backstory="A professional writer who maintains consistency with previously established facts.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("Agents created")

Agents created


## 2. Baseline: Crew Without Memory

First run the workflow with `memory=False`.

In [4]:
task1_no_memory = Task(
    description="Create a fictional AI startup called TechNova. Define its founder, headquarters, mission, and first product.",
    expected_output="A concise company profile for TechNova.",
    agent=researcher
)

task2_no_memory = Task(
    description="Write a follow-up describing TechNova's second-year challenges. Use the company details established by the previous task.",
    expected_output="A concise follow-up analysis.",
    agent=writer,
    context=[task1_no_memory]
)

crew_no_memory = Crew(
    agents=[researcher, writer],
    tasks=[task1_no_memory, task2_no_memory],
    process=Process.sequential,
    memory=False,
    verbose=True
)

print("No-memory crew created")

No-memory crew created


In [5]:
print("RUNNING WITHOUT MEMORY")

result_no_memory = await crew_no_memory.kickoff_async()

print("\nNO-MEMORY RESULT")
print(result_no_memory)

RUNNING WITHOUT MEMORY


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4fcc1ecb-10bb-4b0e-8c42-82683f122419                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a fictional AI startup called TechNova. Define its founder, headquarters, mission, and first      │
│  product.                                                                                                       │
│  ID: 8841b909-6a67-451e-bf45-2fdf3a45a7df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Context-Aware Researcher                                                                                │
│                                                                                                                 │
│  Task: Create a fictional AI startup called TechNova. Define its founder, headquarters, mission, and first      │
│  product.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Context-Aware Researcher                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Company Profile: TechNova**                                                                                  │
│                                                                                                                 │
│  **Founder:** Dr. Emily Carter                                                                                  │
│  Dr. Emily Carter is an accomplished AI researcher with a PhD in Computer Science from MIT. With over a decade  │
│  of experience in the AI sector, Dr. Carter has worked with leading tech companies and has published numerous   │
│  papers on machine learning and natural language processing. She founded TechNova with the vision of making AI  │
│  more accessible and beneficial for everyday users.                                                             │
│                                                                                                                 │
│  **Headquarters:** San Francisco, California                                                                    │
│  TechNova is strategically located in the heart of Silicon Valley, allowing easy access to cutting-edge         │
│  technology, venture capital, and a rich community of innovators and entrepreneurs. The headquarters features   │
│  a collaborative workspace designed to foster creativity and teamwork among its diverse team of engineers,      │
│  designers, and researchers.                                                                                    │
│                                                                                                                 │
│  **Mission:**                                                                                                   │
│  TechNova's mission is to democratize artificial intelligence by creating user-friendly and intuitive           │
│  AI-driven solutions that empower individuals and small businesses. The company aims to enhance productivity    │
│  and creativity while ensuring ethical AI practices and maintaining user privacy and security.                  │
│                                                                                                                 │
│  **First Product:**                                                                                             │
│  TechNova's first product, "NovaAssistant," is an intelligent personal assistant powered by advanced natural    │
│  language processing capabilities. NovaAssistant is designed to manage tasks, schedule appointments, and        │
│  provide relevant information tailored to the user's preferences and needs. Its unique selling point is its     │
│  ability to learn from user interactions, improving its effectiveness over time. NovaAssistant also             │
│  prioritizes user privacy, ensuring data security and offering transparent AI processes.                        │
│                                                                                                                 │
│  TechNova is poised to become a leader in the AI landscape by emphasizing accessibility, innovation, and        │
│  ethical standards in technology.                                                                               │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

... (5 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a follow-up describing TechNova's second-year challenges. Use the company details established by   │
│  the previous task.                                                                                             │
│  Agent: Contextual Writer                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4fcc1ecb-10bb-4b0e-8c42-82683f122419                                                                       │
│  Final Output: As TechNova enters its second year of operation, the company faces several challenges that       │
│  could potentially hinder its trajectory towards achieving its mission of democratizing artificial              │
│  intelligence.                                                                                                  │
│                                                                                                                 │
│  One significant challenge is the increasing competition in the AI market. As major tech companies intensify    │
│  their investments in AI, the landscape is becoming crowded with products that often boast similar features to  │
│  NovaAssistant. This competition not only places pressure on TechNova to continually innovate but also raises   │
│  the stakes for brand visibility and user acquisition. Customers often exhibit brand loyalty towards            │
│  established names, making it imperative for TechNova to find unique ways to distinctively position             │
│  NovaAssistant and communicate its advantages.                                                                  │
│                                                                                                                 │
│  Furthermore, as TechNova expands its user base, it faces the intricate task of scaling its technology while    │
│  maintaining the performance and personalization users have come to expect. The challenge lies in ensuring      │
│  that the AI's learning algorithms remain efficient and effective with a growing volume of data and diverse     │
│  user interactions. TechNova must invest in robust infrastructure and talent capable of managing this growth,   │
│  which could strain finances if not carefully planned.                                                          │
│                                                                                                                 │
│  Additionally, as TechNova emphasizes ethical AI practices, the company must stay ahead of regulatory           │
│  developments and public scrutiny related to data privacy and security. With growing awareness and concern      │
│  around how personal data is handled, TechNova's commitment to transparency and user privacy needs to be        │
│  consistently communicated and upheld. Failure to do so could lead to reputational damage and trust erosion,    │
│  which are critical for a company that relies on user engagement and satisfaction.                              │
│                                                                                                                 │
│  Finally, retaining top talent poses a challenge as TechNova competes with larger tech firms that offer more    │
│  extensive resources and higher compensation packages. Maintaining a culture of innovation within the           │
│  organization, while attracting and retaining skilled professionals, is essential to sustaining TechNova's      │
│  vision and mission.                                                                                            │
│                                                                                                                 │
│  In summary, TechNova's second year presents a complex array of challenges: navigating fierce competition,      │
│  scaling operations efficiently, safeguarding user dat


NO-MEMORY RESULT
As TechNova enters its second year of operation, the company faces several challenges that could potentially hinder its trajectory towards achieving its mission of democratizing artificial intelligence. 

One significant challenge is the increasing competition in the AI market. As major tech companies intensify their investments in AI, the landscape is becoming crowded with products that often boast similar features to NovaAssistant. This competition not only places pressure on TechNova to continually innovate but also raises the stakes for brand visibility and user acquisition. Customers often exhibit brand loyalty towards established names, making it imperative for TechNova to find unique ways to distinctively position NovaAssistant and communicate its advantages.

Furthermore, as TechNova expands its user base, it faces the intricate task of scaling its technology while maintaining the performance and personalization users have come to expect. The challenge lies in

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Memory-Enabled Crew

Now enable CrewAI memory with `memory=True`.

In [6]:
task1_memory = Task(
    description="Create a fictional AI startup called TechNova. Define its founder, headquarters, mission, and first product.",
    expected_output="A concise company profile for TechNova.",
    agent=researcher
)

task2_memory = Task(
    description="Ask a contextual follow-up: what challenges might TechNova face in its second year? Reuse the company facts from the earlier work.",
    expected_output="A concise contextual analysis.",
    agent=writer,
    context=[task1_memory]
)

crew_with_memory = Crew(
    agents=[researcher, writer],
    tasks=[task1_memory, task2_memory],
    process=Process.sequential,
    memory=True,
    verbose=True
)

print("Memory-enabled crew created")

Memory-enabled crew created


In [7]:
print("RUNNING WITH MEMORY")

result_with_memory = await crew_with_memory.kickoff_async()

print("\nMEMORY-ENABLED RESULT")
print(result_with_memory)

RUNNING WITH MEMORY


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f6e51818-0b2f-432a-a70f-2a780cb175d1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a fictional AI startup called TechNova. Define its founder, headquarters, mission, and first      │
│  product.                                                                                                       │
│  ID: e8a63835-9f36-4362-8b53-3ec4ca99a7b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Context-Aware Researcher                                                                                │
│                                                                                                                 │
│  Task: Create a fictional AI startup called TechNova. Define its founder, headquarters, mission, and first      │
│  product.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: save_to_memory                                                                                           │
│  Args: {'contents': ['TechNova is a fictional AI startup.', 'The founder of TechNova is Alex Chen.',            │
│  "TechNova's headquarters is located in San Francisco, California.", 'The mission of TechNova is to democra...  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['TechNova', 'Alex Chen', 'San Francisco', 'AI technologies for small businesses',           │
│  'InsightIQ']}                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (13 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Ask a contextual follow-up: what challenges might TechNova face in its second year? Reuse the company    │
│  facts from the earlier work.                                                                                   │
│  Agent: Contextual Writer                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f6e51818-0b2f-432a-a70f-2a780cb175d1                                                                       │
│  Final Output: As TechNova enters its second year, it may encounter several challenges that could impact its    │
│  growth and ability to fulfill its mission of democratizing access to advanced AI technologies for small        │
│  businesses.                                                                                                    │
│                                                                                                                 │
│  One primary obstacle could be competition. The market for AI solutions is rapidly evolving, and as TechNova    │
│  has positioned itself to serve small businesses with its flagship product, InsightIQ, it may face increased    │
│  competition from both established players and new startups that could offer similar or enhanced                │
│  functionalities at competitive prices.                                                                         │
│                                                                                                                 │
│  Additionally, scaling operations and customer support to meet the growing demand might pose a challenge.       │
│  TechNova will need to ensure that its infrastructure can support an expanding user base while maintaining      │
│  high levels of service. This includes effective onboarding and ongoing support for small businesses that may   │
│  be new to AI technology.                                                                                       │
│                                                                                                                 │
│  Funding and financial management will also be critical in the second year. Depending on its initial traction   │
│  and revenue generation from InsightIQ, TechNova may need to secure additional investments to scale its         │
│  operations, enhance its product offerings, and penetrate new markets more effectively.                         │
│                                                                                                                 │
│  Finally, TechNova must continue to innovate and adapt its technologies to meet the evolving needs of small     │
│  businesses. Keeping pace with technological advancements and customer feedback will be paramount to retain     │
│  relevance and maintain a competitive edge.                                                                     │
│                                                                                                                 │
│  By addressing these challenges proactively, TechNova can enhance its prospects as it aims to empower small     │
│  businesses with cutting-edge AI solutions.                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


MEMORY-ENABLED RESULT
As TechNova enters its second year, it may encounter several challenges that could impact its growth and ability to fulfill its mission of democratizing access to advanced AI technologies for small businesses.

One primary obstacle could be competition. The market for AI solutions is rapidly evolving, and as TechNova has positioned itself to serve small businesses with its flagship product, InsightIQ, it may face increased competition from both established players and new startups that could offer similar or enhanced functionalities at competitive prices.

Additionally, scaling operations and customer support to meet the growing demand might pose a challenge. TechNova will need to ensure that its infrastructure can support an expanding user base while maintaining high levels of service. This includes effective onboarding and ongoing support for small businesses that may be new to AI technology.

Funding and financial management will also be critical in the second

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Compare Memory Workflows

| Feature | Without Memory | With Memory |
|---|---|---|
| Context retention | Limited to supplied task context | Crew memory enabled |
| Reuse of information | Explicit context | Memory can retain useful information |
| Follow-up questions | More dependent on task context | Better suited to repeated interactions |
| Workflow continuity | Lower | Higher |

`context=[previous_task]` passes context within a workflow. `memory=True` enables CrewAI's memory system.

## 5. Create a Knowledge Document

Knowledge is different from conversation memory. Here we create a company document that an agent can read.

In [8]:
knowledge_doc = """TechNova Company Profile

Founded: 2023
Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)
Headquarters: Austin, Texas
Mission: Democratize AI tools for small businesses

Products:
1. NovaAI - AI-powered business assistant
2. NovaAnalytics - Data analysis platform
3. NovaChat - Customer service chatbot

Funding: $12M Series A (2024)
Team Size: 45 employees
"""

with open("technova_knowledge.txt", "w", encoding="utf-8") as f:
    f.write(knowledge_doc)

print("Knowledge document created")

Knowledge document created


## 6. Create Knowledge-Reading Tool

In [9]:
@tool
def read_technova_knowledge() -> str:
    """Read the TechNova knowledge document."""
    try:
        with open("technova_knowledge.txt", "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        return f"Error reading knowledge document: {e}"

print(read_technova_knowledge.run())

TechNova Company Profile

Founded: 2023
Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)
Headquarters: Austin, Texas
Mission: Democratize AI tools for small businesses

Products:
1. NovaAI - AI-powered business assistant
2. NovaAnalytics - Data analysis platform
3. NovaChat - Customer service chatbot

Funding: $12M Series A (2024)
Team Size: 45 employees



## 7. Create Knowledge Agent

In [10]:
knowledge_agent = Agent(
    role="Company Knowledge Analyst",
    goal="Answer questions accurately using the TechNova knowledge document",
    backstory="An analyst who carefully reads company documentation and answers from available information.",
    llm=llm,
    tools=[read_technova_knowledge],
    verbose=True,
    allow_delegation=False
)

print("Knowledge agent created")

Knowledge agent created


## 8. Ask Contextual Knowledge Questions

In [11]:
knowledge_task = Task(
    description=(
        "Read the TechNova knowledge document. Answer: who founded TechNova, "
        "where is it headquartered, what is its mission, what products does it offer, "
        "and how much funding has it raised?"
    ),
    expected_output="A concise factual answer containing founders, location, mission, products, and funding.",
    agent=knowledge_agent
)

knowledge_crew = Crew(
    agents=[knowledge_agent],
    tasks=[knowledge_task],
    process=Process.sequential,
    memory=False,
    verbose=True
)

knowledge_result = await knowledge_crew.kickoff_async()

print("\nKNOWLEDGE RESULT")
print(knowledge_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 12712366-fe58-49ee-b246-5ac6895c7046                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the TechNova knowledge document. Answer: who founded TechNova, where is it headquartered, what is   │
│  its mission, what products does it offer, and how much funding has it raised?                                  │
│  ID: 4544fcd6-d7bb-404c-aaa4-76da24a56533                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Company Knowledge Analyst                                                                               │
│                                                                                                                 │
│  Task: Read the TechNova knowledge document. Answer: who founded TechNova, where is it headquartered, what is   │
│  its mission, what products does it offer, and how much funding has it raised?                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_technova_knowledge                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_technova_knowledge executed with result: TechNova Company Profile

Founded: 2023
Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)
Headquarters: Austin, Texas
Mission: Democratize AI tools for small businesses

Products:
1. NovaAI - AI-p...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: read_technova_knowledge                                                                                  │
│  Output: TechNova Company Profile                                                                               │
│                                                                                                                 │
│  Founded: 2023                                                                                                  │
│  Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)                                                          │
│  Headquarters: Austin, Texas                                                                                    │
│  Mission: Democratize AI tools for small businesses                                                             │
│                                                                                                                 │
│  Products:                                                                                                      │
│  1. NovaAI - AI-powered business assistant                                                                      │
│  2. NovaAnalytics - Data analysis platform                                                                      │
│  3. NovaChat - Customer service chatbot                                                                         │
│                                                                                                                 │
│  Funding: $12M Series A (2024)                                                                                  │
│  Team Size: 45 employees                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (2 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Read the TechNova knowledge document. Answer: who founded TechNova, where is it headquartered, what is   │
│  its mission, what products does it offer, and how much funding has it raised?                                  │
│  Agent: Company Knowledge Analyst                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 12712366-fe58-49ee-b246-5ac6895c7046                                                                       │
│  Final Output: TechNova Company Profile                                                                         │
│                                                                                                                 │
│  Founded: 2023                                                                                                  │
│  Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)                                                          │
│  Headquarters: Austin, Texas                                                                                    │
│  Mission: Democratize AI tools for small businesses                                                             │
│                                                                                                                 │
│  Products:                                                                                                      │
│  1. NovaAI - AI-powered business assistant                                                                      │
│  2. NovaAnalytics - Data analysis platform                                                                      │
│  3. NovaChat - Customer service chatbot                                                                         │
│                                                                                                                 │
│  Funding: $12M Series A (2024)                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


KNOWLEDGE RESULT
TechNova Company Profile

Founded: 2023  
Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)  
Headquarters: Austin, Texas  
Mission: Democratize AI tools for small businesses  

Products:  
1. NovaAI - AI-powered business assistant  
2. NovaAnalytics - Data analysis platform  
3. NovaChat - Customer service chatbot  

Funding: $12M Series A (2024)  


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 9. Reuse Previous Information

Run a follow-up task using the same knowledge source.

In [13]:
followup_task = Task(
    description=(
        "Using the TechNova knowledge document, answer this follow-up: "
        "Which product is most directly related to customer service, and what is the current team size?"
    ),
    expected_output="A short factual follow-up answer.",
    agent=knowledge_agent
)

followup_crew = Crew(
    agents=[knowledge_agent],
    tasks=[followup_task],
    process=Process.sequential,
    memory=True,
    verbose=True
)

followup_result = await followup_crew.kickoff_async()

print("\nFOLLOW-UP RESULT")
print(followup_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 98407a76-4499-44f0-be21-52435e09e817                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Using the TechNova knowledge document, answer this follow-up: Which product is most directly related to  │
│  customer service, and what is the current team size?                                                           │
│  ID: c276f122-868a-499b-a5b7-2800dd4a3662                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Company Knowledge Analyst                                                                               │
│                                                                                                                 │
│  Task: Using the TechNova knowledge document, answer this follow-up: Which product is most directly related to  │
│  customer service, and what is the current team size?                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['customer service product', 'TechNova product related to customer service', 'current team   │
│  size', 'TechNova team size']}                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Found memories:
- (score=0.66) TechNova's first product is an AI-driven business analytics platform called InsightIQ.
  categories: product, AI, business analytics
  entities: ['TechNova', 'InsightIQ'...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Output: Found memories:                                                                                        │
│  - (score=0.66) TechNova's first product is an AI-driven business analytics platform called InsightIQ.          │
│    categories: product, AI, business analytics                                                                  │
│    entities: ['TechNova', 'InsightIQ']                                                                          │
│    dates: []                                                                                                    │
│    topics: ['AI', 'business analytics', 'product development']                                                  │
│  - (score=0.64) The mission of TechNova is to democratize access to advanced AI technologies for small          │
│  businesses.                                                                                                    │
│    categories: mission, technology, AI, small business                                                          │
│    entities: ['TechNova']                                                                                       │
│    dates: []                                                                                                    │
│    topics: ['AI technologies', 'access', 'democratization', 'small businesses']                                 │
│  - (score=0.63) TechNova's headquarters is located in San Francisco, California.                                │
│    categories: company, headquarters, location                                                                  │
│    entities: ['TechNova', 'San Francisco', 'California']                                                        │
│    dates: []                                                                                                    │
│    topics: ['company locations', 'headquarters']                                                                │
│  - (score=0.62) TechNova is a fictional AI startup.                                                             │
│    categories: startup, AI                                                                                      │
│    entities: ['TechNova']                                                                                       │
│    dates: []                                                                                                    │
│    topics: ['artificial intelligence', 'startups']                                                              │
│  - (score=0.62) The founder of TechNova is Alex Chen.                                                           │
│    categories: company, founders                                                                                │
│    entities: ['TechNova', 'Alex Chen']                                                                          │
│    dates: []                                                                                                    │
│    topics: ['foundership', 'technology', 'startups']                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (10 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the TechNova knowledge document, answer this follow-up: Which product is most directly related to  │
│  customer service, and what is the current team size?                                                           │
│  Agent: Company Knowledge Analyst                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 98407a76-4499-44f0-be21-52435e09e817                                                                       │
│  Final Output: TechNova's product most directly related to customer service is NovaChat, a customer service     │
│  chatbot. TechNova currently has a team size of 45 employees.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FOLLOW-UP RESULT
TechNova's product most directly related to customer service is NovaChat, a customer service chatbot. TechNova currently has a team size of 45 employees.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 10. Final Verification

- ✅ Crew memory configured
- ✅ Memory-disabled baseline tested
- ✅ Memory-enabled workflow tested
- ✅ Knowledge document created
- ✅ Knowledge-reading tool created
- ✅ Contextual questions tested
- ✅ Previous information reused
- ✅ Memory and knowledge compared

### Key difference

**Memory →** retains useful workflow information.

**Knowledge →** provides factual information from documents.